In [ ]:
import time
import json
import requests
from io import StringIO
from azure.kusto.data import KustoConnectionStringBuilder
from azure.kusto.data.data_format import DataFormat
from azure.kusto.ingest import QueuedIngestClient as KustoIngestClient, IngestionProperties

# ==============================================================================
# CONFIGURATION
# ==============================================================================
# 1. Ingestion URI from Fabric
INGEST_URI = ""

# 2. KQL Database name
KQL_DATABASE = "Aviation_Eventhouse"

# 3.Tabele name
KQL_TABLE = "RawFlights"
# ==============================================================================

# Lat. and Long. for Poland
POLAND_BOUNDS = {"lamin": 49.0, "lamin_max": 55.0, "lomin": 14.0, "lomax": 24.0}


def fetch_flights_over_poland():
    """Actuals flights - Poland API OpenSky."""
    url = "https://opensky-network.org/api/states/all"
    params = {
        "lamin": POLAND_BOUNDS["lamin"],
        "lamax": POLAND_BOUNDS["lamin_max"],
        "lomin": POLAND_BOUNDS["lomin"],
        "lomax": POLAND_BOUNDS["lomax"],
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            states = data.get("states", [])
            current_timestamp = data.get("time")

            flights = []
            if states:
                for s in states:
                    flight_record = {
                        "icao24": s[0],
                        "callsign": s[1].strip() if s[1] else "UNKNOWN",
                        "country": s[2],
                        "longitude": float(s[5]) if s[5] is not None else 0.0,
                        "latitude": float(s[6]) if s[6] is not None else 0.0,
                        "altitude": float(s[7]) if s[7] is not None else 0.0,
                        "velocity": float(s[9]) if s[9] is not None else 0.0,
                        "timestamp": time.strftime(
                            "%Y-%m-%dT%H:%M:%SZ", time.gmtime(current_timestamp)
                        ),
                    }
                    flights.append(flight_record)
            return flights
        else:
            print(f"Błąd API OpenSky: Status {response.status_code}")
            return []
    except Exception as e:
        print(f"Nie udało się pobrać danych: {e}")
        return []


def main():
   
    print("Connection with Microsoft Fabric KQL...")
    kcsb = KustoConnectionStringBuilder.with_interactive_login(INGEST_URI)
    ingest_client = KustoIngestClient(kcsb)

    ingestion_properties = IngestionProperties(
        database=KQL_DATABASE, table=KQL_TABLE, data_format=DataFormat.MULTIJSON
    )

    print("Data transfer started. Press Ctrl+C to abort.")

    while True:
        flights_data = fetch_flights_over_poland()

        if flights_data:
            print(f"Pobrano {len(flights_data)} lotów. Wysyłanie do Fabric...")

            # Converting data to MultiJSON format (each object on a new line)
            json_lines = "\n".join([json.dumps(f) for f in flights_data])
            stream = StringIO(json_lines)

            try:
                # Injecting a data stream directly into the table
                ingest_client.ingest_from_stream(
                    stream, ingestion_properties=ingestion_properties
                )
                print(" -> Data sent successfully!")
            except Exception as e:
                print(f" -> Błąd podczas wysyłania danych do Fabric: {e}")
        else:
            print("There are no current flights over the specified area at this second.")

        # Wait 15 seconds before downloading again
        time.sleep(15)


if __name__ == "__main__":
    main()

In [1]:
pip install azure-kusto-data azure-kusto-ingest requests

  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.0.0
    Uninstalling tenacity-9.0.0:
      Successfully uninstalled tenacity-9.0.0
  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Uninstalling requests-2.32.3:
      Successfully uninstalled requests-2.32.3
  Attempting uninstall: azure-storage-blob
    Found existing installation: azure-storage-blob 12.28.0
    Uninstalling azure-storage-blob-12.28.0:
      Successfully uninstalled azure-storage-blob-12.28.0
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aext-assistant-server 4.1.0 requires anaconda-cloud-auth>=0.7.1, which is not installed.
azure-storage-file-datalake 12.23.0 requires azure-storage-blob>=12.28.0, but you have azure-storage-blob 12.26.0 which is incompatible.
dash 2.18.2 requires Flask<3.1,>=1.0.4, but you have flask 3.1.0 which is incompatible.
dash 2.18.2 requires Werkzeug<3.1, but you have werkzeug 3.1.3 which is incompatible.
